# Stage 1: Exploratory Data Analysis & MSME Proxy Definition

**Thesis:** Predictive Analytics for MSME Credit Risk Assessment using Behavioural Feature Engineering and Explainable Ensemble Machine Learning

Notebook 1 of 5. Before doing anything with behavioural features I need to get a
feel for the raw data, and more importantly work out how to even identify MSME-type
borrowers in a dataset that doesn't label them directly. So this notebook is mostly
groundwork:

1. Load the main application table and get a sense of its shape
2. Check how imbalanced the classes are
3. Work out and justify an MSME **proxy** (business-owner applicants)
4. Look at missingness for that proxy population
5. Compare a few key attributes between defaulters and non-defaulters
6. Look at the external credit scores (`EXT_SOURCE_*`)
7. Check simple linear correlations with the target
8. See how well the MSME proxy is covered across the relational sub-tables (needed for Stage 2)

Figures get saved to `outputs/` at 300 dpi so they can go straight into the thesis.
The MSME proxy population is written out to `outputs/msme_proxy.csv` for the rest
of the notebooks to use.


In [1]:

# Setup - imports, paths, a fixed seed, and a plotting style I'll reuse everywhere
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # don't try to render inline, just save to file
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
plt.ioff()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ROOT = the project folder, whether I run this from inside notebooks/ or from
# the project root itself
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(ROOT, "home-credit-default-risk")
OUT_DIR = os.path.join(ROOT, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# a clean, colour-blind-friendly style so the figures look consistent in the thesis
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

# same two colours for repaid/default everywhere so figures stay comparable
C_REPAID  = "#4C72B0"   # TARGET = 0
C_DEFAULT = "#C44E52"   # TARGET = 1
PALETTE = {0: C_REPAID, 1: C_DEFAULT}

FIG_INDEX = {}   # filename -> caption, so I can dump a List-of-Figures index at the end

def savefig(fig, name, caption=""):
    """Save a figure into outputs/ at 300 dpi and keep track of its caption."""
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX[name] = caption
    print(f"saved -> {os.path.relpath(path, ROOT)}" + (f"   |  {caption}" if caption else ""))
    return path

print("Project root :", ROOT)
print("Data dir     :", DATA_DIR)
print("Output dir    :", OUT_DIR)


Project root : C:\Users\VARUN\OneDrive\Desktop\Final Report
Data dir     : C:\Users\VARUN\OneDrive\Desktop\Final Report\home-credit-default-risk
Output dir    : C:\Users\VARUN\OneDrive\Desktop\Final Report\outputs


## 1. Load the main application table

`application_train.csv` has one row per loan application, with the binary target
`TARGET` (1 = had payment difficulties / defaulted, 0 = repaid) plus 121 other
columns describing the applicant.


In [2]:
app = pd.read_csv(os.path.join(DATA_DIR, "application_train.csv"))
print(f"application_train shape : {app.shape[0]:,} rows  x  {app.shape[1]} columns")

# quick breakdown of column types
n_num = app.select_dtypes(include="number").shape[1]
n_cat = app.select_dtypes(include="object").shape[1]
print(f"numeric columns        : {n_num}")
print(f"categorical columns    : {n_cat}")
print(f"in-memory size         : {app.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

# just the columns I actually care about for this study
app[["SK_ID_CURR", "TARGET", "NAME_INCOME_TYPE", "ORGANIZATION_TYPE",
     "OCCUPATION_TYPE", "AMT_INCOME_TOTAL", "AMT_CREDIT", "DAYS_BIRTH"]].head()


application_train shape : 307,511 rows  x  122 columns
numeric columns        : 106
categorical columns    : 16
in-memory size         : 341 MB


,SK_ID_CURR,TARGET,NAME_INCOME_TYPE,ORGANIZATION_TYPE,OCCUPATION_TYPE,AMT_INCOME_TOTAL,AMT_CREDIT,DAYS_BIRTH
0,100002,1,Working,Business Entity Type 3,Laborers,202500.0,406597.5,-9461
1,100003,0,State servant,School,Core staff,270000.0,1293502.5,-16765
2,100004,0,Working,Government,Laborers,67500.0,135000.0,-19046
3,100006,0,Working,Business Entity Type 3,Laborers,135000.0,312682.5,-19005
4,100007,0,Working,Religion,Core staff,121500.0,513000.0,-19932


## 2. Class imbalance in the full lending population

Credit-default data is always going to be imbalanced - defaults are the minority
class by a good margin. This is why SMOTE is needed on the training partition
later (Stage 3), and why **default-class recall** rather than plain accuracy is
the metric I actually care about (Stage 4).


In [3]:
counts = app["TARGET"].value_counts().sort_index()
pct = app["TARGET"].value_counts(normalize=True).sort_index() * 100
imb_ratio = counts[0] / counts[1]

print("Full population (n = {:,})".format(len(app)))
print(f"  Repaid   (TARGET=0): {counts[0]:>7,}  ({pct[0]:5.2f}%)")
print(f"  Default  (TARGET=1): {counts[1]:>7,}  ({pct[1]:5.2f}%)")
print(f"  Imbalance ratio    : {imb_ratio:.1f} : 1")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Repaid\n(TARGET = 0)", "Default\n(TARGET = 1)"], counts.values,
              color=[C_REPAID, C_DEFAULT], width=0.6, edgecolor="black", linewidth=0.6)
for b, n, p in zip(bars, counts.values, pct.values):
    ax.text(b.get_x() + b.get_width()/2, n + 4000, f"{n:,}\n({p:.1f}%)",
            ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Number of loan applications")
ax.set_title("Class imbalance in the full lending population")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.set_ylim(0, counts.max() * 1.18)
sns.despine(ax=ax)
savefig(fig, "eda_01_class_imbalance_full.png",
        "Class distribution of the target variable in the full application population "
        f"(n = {len(app):,}); defaults represent {pct[1]:.1f}% of cases.")


Full population (n = 307,511)
  Repaid   (TARGET=0): 282,686  (91.93%)
  Default  (TARGET=1):  24,825  ( 8.07%)
  Imbalance ratio    : 11.4 : 1


saved -> outputs\eda_01_class_imbalance_full.png   |  Class distribution of the target variable in the full application population (n = 307,511); defaults represent 8.1% of cases.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\eda_01_class_imbalance_full.png'

## 3. Defining the MSME proxy sub-population

Home Credit doesn't label MSME borrowers directly, so I need a proxy. The plan is
to identify business-owner applicants using occupation-type and organisation-type
flags already in the main table. I tried three candidate signals:

| Candidate | Field | Rationale | Limitation |
|---|---|---|---|
| A | `NAME_INCOME_TYPE == "Businessman"` | Explicit business owner | Tiny sample |
| B | `ORGANIZATION_TYPE == "Self-employed"` | Runs own micro/small business; not on an employer payroll | Some may be sole traders rather than employers |
| C | `OCCUPATION_TYPE` in managerial / proprietor roles | Captures owner-managers | Managers can be salaried employees of large firms |

Whichever one best isolates genuine small-business owners **while still leaving
enough data to model** is the one I'll go with.


In [4]:
# Candidate A: explicit "Businessman" income type
a = app[app["NAME_INCOME_TYPE"] == "Businessman"]
print(f"A  NAME_INCOME_TYPE == 'Businessman' : n = {len(a):,}  "
      f"default rate = {a['TARGET'].mean():.3f}")

# Candidate B: self-employed organisation type
b = app[app["ORGANIZATION_TYPE"] == "Self-employed"]
print(f"B  ORGANIZATION_TYPE == 'Self-employed': n = {len(b):,}  "
      f"default rate = {b['TARGET'].mean():.3f}")

# Candidate C: owner-manager occupation types
owner_occ = ["Managers", "Realty agents", "Private service staff"]
c = app[app["OCCUPATION_TYPE"].isin(owner_occ)]
print(f"C  OCCUPATION_TYPE in {owner_occ}: n = {len(c):,}  "
      f"default rate = {c['TARGET'].mean():.3f}")

print(f"\nFull-population default rate            : {app['TARGET'].mean():.3f}")
print(f"Overlap B ∩ A (self-employed businessmen): {len(b.index.intersection(a.index)):,} of {len(a)}")
print(f"Overlap B ∩ C                            : {len(b.index.intersection(c.index)):,}")


A  NAME_INCOME_TYPE == 'Businessman' : n = 10  default rate = 0.000
B  ORGANIZATION_TYPE == 'Self-employed': n = 38,412  default rate = 0.102


C  OCCUPATION_TYPE in ['Managers', 'Realty agents', 'Private service staff']: n = 24,774  default rate = 0.063



Full-population default rate            : 0.081
Overlap B ∩ A (self-employed businessmen): 8 of 10
Overlap B ∩ C                            : 4,648


In [5]:
# default rate by organisation type, with Self-employed picked out
org = (app.groupby("ORGANIZATION_TYPE")
          .agg(n=("TARGET", "size"), default_rate=("TARGET", "mean"))
          .query("n >= 2000")
          .sort_values("default_rate", ascending=False))

fig, ax = plt.subplots(figsize=(7, 8))
colors = ["#C44E52" if t == "Self-employed" else "#B0B7C0" for t in org.index]
ax.barh(org.index, org["default_rate"] * 100, color=colors, edgecolor="black", linewidth=0.4)
ax.axvline(app["TARGET"].mean() * 100, color="#333333", ls="--", lw=1,
           label=f"full-population rate ({app['TARGET'].mean()*100:.1f}%)")
ax.set_xlabel("Default rate (%)")
ax.set_title("Default rate by organisation type\n(types with n ≥ 2,000; Self-employed highlighted)")
ax.invert_yaxis()
ax.legend(loc="lower right", frameon=True)
sns.despine(ax=ax)
savefig(fig, "eda_02_default_rate_by_organisation.png",
        "Default rate by organisation type for categories with at least 2,000 applicants. "
        "The Self-employed group (highlighted) is the MSME proxy adopted for this study.")


saved -> outputs\eda_02_default_rate_by_organisation.png   |  Default rate by organisation type for categories with at least 2,000 applicants. The Self-employed group (highlighted) is the MSME proxy adopted for this study.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\eda_02_default_rate_by_organisation.png'

In [6]:
# build the MSME proxy population
msme = app[app["ORGANIZATION_TYPE"] == "Self-employed"].reset_index(drop=True).copy()

m_counts = msme["TARGET"].value_counts().sort_index()
m_pct = msme["TARGET"].value_counts(normalize=True).sort_index() * 100

print("MSME proxy sub-population (ORGANIZATION_TYPE == 'Self-employed')")
print(f"  records            : {len(msme):,}   ({len(msme)/len(app)*100:.1f}% of full population)")
print(f"  Repaid  (TARGET=0)  : {m_counts[0]:>6,}  ({m_pct[0]:.2f}%)")
print(f"  Default (TARGET=1)  : {m_counts[1]:>6,}  ({m_pct[1]:.2f}%)")
print(f"  imbalance ratio     : {m_counts[0]/m_counts[1]:.1f} : 1")

# save it so the other notebooks don't have to redo this step
msme_path = os.path.join(OUT_DIR, "msme_proxy.csv")
msme.to_csv(msme_path, index=False)
print(f"\nsaved -> {msme_path}  ({os.path.getsize(msme_path)/1e6:.1f} MB)")


MSME proxy sub-population (ORGANIZATION_TYPE == 'Self-employed')
  records            : 38,412   (12.5% of full population)
  Repaid  (TARGET=0)  : 34,504  (89.83%)
  Default (TARGET=1)  :  3,908  (10.17%)
  imbalance ratio     : 8.8 : 1



saved -> C:\Users\VARUN\OneDrive\Desktop\Final Report\outputs\msme_proxy.csv  (20.2 MB)


## 4. Missing-value profile of the MSME proxy

From here on everything is done on the MSME proxy only, since that's what actually
gets modelled. I'm grouping features by roughly why they're missing, so Stage 3
can pick a sensible imputation strategy instead of a one-size-fits-all approach.


In [7]:
# DAYS_EMPLOYED uses 365243 as a placeholder for "not currently employed" - that's
# obviously not a real day count, so treat it as missing instead
msme["DAYS_EMPLOYED"] = msme["DAYS_EMPLOYED"].replace(365243, np.nan)

miss = (msme.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
print(f"{len(miss)} of {msme.shape[1]} columns contain missing values "
      f"(range {miss.min():.1f}% – {miss.max():.1f}%)\n")
print("Top 15 by missing share:")
print(miss.head(15).round(1).to_string())

top = miss.head(25)
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(top.index, top.values, color="#6E8CA0", edgecolor="black", linewidth=0.4)
ax.set_xlabel("Missing (%)")
ax.set_title("Top 25 features by missing-value share (MSME proxy)")
ax.invert_yaxis()
sns.despine(ax=ax)
savefig(fig, "eda_03_missingness_top25.png",
        "The 25 features with the highest proportion of missing values in the MSME "
        "proxy sub-population. Building-information and external-score fields dominate.")


65 of 122 columns contain missing values (range 0.0% – 73.8%)

Top 15 by missing share:
COMMONAREA_AVG              73.8
COMMONAREA_MODE             73.8
COMMONAREA_MEDI             73.8
NONLIVINGAPARTMENTS_MEDI    73.4
NONLIVINGAPARTMENTS_MODE    73.4
NONLIVINGAPARTMENTS_AVG     73.4
FONDKAPREMONT_MODE          72.3
LIVINGAPARTMENTS_AVG        72.3
LIVINGAPARTMENTS_MEDI       72.3
LIVINGAPARTMENTS_MODE       72.3
FLOORSMIN_MODE              71.9
FLOORSMIN_AVG               71.9
FLOORSMIN_MEDI              71.9
YEARS_BUILD_AVG             70.7
YEARS_BUILD_MODE            70.7


saved -> outputs\eda_03_missingness_top25.png   |  The 25 features with the highest proportion of missing values in the MSME proxy sub-population. Building-information and external-score fields dominate.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\eda_03_missingness_top25.png'

## 5. Key applicant attributes by default status

Comparing four basic attributes between repaid and defaulted loans: age, length of
employment, declared income, and requested credit amount. `DAYS_BIRTH` and
`DAYS_EMPLOYED` come as negative day counts (relative to the application date), so
I convert them to years so they're actually readable.


In [8]:
d = msme.copy()
d["AGE_YEARS"] = -d["DAYS_BIRTH"] / 365.25
d["EMPLOYED_YEARS"] = -d["DAYS_EMPLOYED"] / 365.25

panels = [
    ("AGE_YEARS",        "Applicant age (years)",        None,   False),
    ("EMPLOYED_YEARS",   "Length of employment (years)", (0, 30), False),
    ("AMT_INCOME_TOTAL", "Annual income",                None,   True),
    ("AMT_CREDIT",       "Requested credit amount",      None,   True),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (col, label, xlim, logx) in zip(axes.ravel(), panels):
    for t in (0, 1):
        s = d.loc[d["TARGET"] == t, col].dropna()
        if logx:
            s = np.log10(s[s > 0])
        sns.kdeplot(s, ax=ax, fill=True, alpha=0.35, linewidth=1.4,
                    color=PALETTE[t], label=("Repaid" if t == 0 else "Default"))
    ax.set_title(label)
    ax.set_xlabel(f"log10({label})" if logx else label)
    ax.set_ylabel("Density")
    if xlim:
        ax.set_xlim(*xlim)
    ax.legend()
    sns.despine(ax=ax)
fig.suptitle("Distribution of key applicant attributes by default status (MSME proxy)",
             y=1.02, fontsize=13, fontweight="bold")
fig.tight_layout()
savefig(fig, "eda_04_key_distributions.png",
        "Kernel-density comparison of applicant age, employment length, annual income "
        "(log10) and requested credit amount (log10) between repaid and defaulted loans "
        "in the MSME proxy.")

# medians, to quote directly in the write-up
print(d.groupby("TARGET")[["AGE_YEARS", "EMPLOYED_YEARS", "AMT_INCOME_TOTAL", "AMT_CREDIT"]]
        .median().round(0).to_string())


saved -> outputs\eda_04_key_distributions.png   |  Kernel-density comparison of applicant age, employment length, annual income (log10) and requested credit amount (log10) between repaid and defaulted loans in the MSME proxy.
        AGE_YEARS  EMPLOYED_YEARS  AMT_INCOME_TOTAL  AMT_CREDIT
TARGET                                                         
0            40.0             4.0          150172.0    518562.0
1            37.0             3.0          135000.0    485950.0


## 6. External credit scores (`EXT_SOURCE_1/2/3`)

These three fields are normalised scores from outside data providers. Everyone
who's worked with Home Credit knows they're usually the strongest predictors
around, so it's worth checking how they behave on the MSME proxy specifically
before I go any further.


In [9]:
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

summary = pd.DataFrame({
    "missing_%": (msme[ext_cols].isna().mean() * 100).round(1),
    "corr_with_TARGET": msme[ext_cols].corrwith(msme["TARGET"]).round(3),
})
print(summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ext_cols):
    for t in (0, 1):
        s = msme.loc[msme["TARGET"] == t, col].dropna()
        sns.kdeplot(s, ax=ax, fill=True, alpha=0.35, linewidth=1.4,
                    color=PALETTE[t], label=("Repaid" if t == 0 else "Default"))
    ax.set_title(f"{col}  (missing {summary.loc[col, 'missing_%']:.0f}%)")
    ax.set_xlabel("Score")
    ax.legend()
    sns.despine(ax=ax)
fig.suptitle("External credit-score distributions by default status (MSME proxy)",
             y=1.04, fontsize=13, fontweight="bold")
fig.tight_layout()
savefig(fig, "eda_05_ext_source_distributions.png",
        "Distribution of the three external credit scores by default status in the MSME "
        "proxy. Defaulted applicants show consistently lower scores on all three sources.")


              missing_%  corr_with_TARGET
EXT_SOURCE_1       51.9            -0.163
EXT_SOURCE_2        0.2            -0.182
EXT_SOURCE_3       25.1            -0.187


saved -> outputs\eda_05_ext_source_distributions.png   |  Distribution of the three external credit scores by default status in the MSME proxy. Defaulted applicants show consistently lower scores on all three sources.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\eda_05_ext_source_distributions.png'

## 7. Linear correlation with the target

Just plain Pearson correlations between the numeric main-table features and
`TARGET` - a rough, purely-linear first look at which raw signals actually carry
information. The ensemble models in Stage 4 will pick up non-linear and
interaction effects on top of this, and I'm expecting the behavioural features
from Stage 2 to add signal beyond what shows up here.


In [10]:
num = msme.select_dtypes(include="number").drop(columns=["SK_ID_CURR"])
corr = num.corrwith(num["TARGET"]).drop("TARGET").dropna().sort_values()

topk = pd.concat([corr.head(12), corr.tail(12)])
colors = [C_DEFAULT if v > 0 else C_REPAID for v in topk.values]

fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(topk.index, topk.values, color=colors, edgecolor="black", linewidth=0.4)
ax.axvline(0, color="#333333", lw=0.8)
ax.set_xlabel("Pearson correlation with TARGET")
ax.set_title("Strongest linear correlations with default (MSME proxy)")
ax.invert_yaxis()
sns.despine(ax=ax)
savefig(fig, "eda_06_target_correlation.png",
        "The 12 most negatively and 12 most positively correlated numeric main-table "
        "features with the default indicator in the MSME proxy.")

print("Most positively correlated with default:")
print(corr.tail(8)[::-1].round(3).to_string())
print("\nMost negatively correlated with default:")
print(corr.head(8).round(3).to_string())


saved -> outputs\eda_06_target_correlation.png   |  The 12 most negatively and 12 most positively correlated numeric main-table features with the default indicator in the MSME proxy.
Most positively correlated with default:
DAYS_EMPLOYED                  0.073
REGION_RATING_CLIENT           0.070
DAYS_LAST_PHONE_CHANGE         0.069
REGION_RATING_CLIENT_W_CITY    0.069
DAYS_BIRTH                     0.061
DEF_30_CNT_SOCIAL_CIRCLE       0.045
DEF_60_CNT_SOCIAL_CIRCLE       0.044
DAYS_ID_PUBLISH                0.044

Most negatively correlated with default:
EXT_SOURCE_3     -0.187
EXT_SOURCE_2     -0.182
EXT_SOURCE_1     -0.163
FLOORSMIN_AVG    -0.059
FLOORSMIN_MEDI   -0.058
FLOORSMAX_AVG    -0.056
FLOORSMIN_MODE   -0.056
FLOORSMAX_MEDI   -0.056


## 8. Relational sub-table coverage for the MSME proxy

Stage 2 builds behavioural features out of the relational sub-tables, so before I
get there it's worth checking how many of the 38,412 MSME-proxy applicants
actually show up in each sub-table, and roughly how many rows (repayments,
monthly balances, previous applications) are typically available per applicant.
I'm scanning these in chunks so memory doesn't blow up on the bigger files.
`bureau_balance` only links in through `bureau`, so its coverage has to be worked
out via the bureau credit IDs rather than directly.


In [11]:
msme_ids = set(msme["SK_ID_CURR"])
n_msme = len(msme_ids)
CHUNK = 1_000_000

def scan_counts(filename, key="SK_ID_CURR", id_set=msme_ids):
    """Read a sub-table in chunks and count rows per matching id, without ever
    loading the whole file into memory at once."""
    acc = pd.Series(dtype="int64")
    for ch in pd.read_csv(os.path.join(DATA_DIR, filename), usecols=[key], chunksize=CHUNK):
        ch = ch[ch[key].isin(id_set)]
        if len(ch):
            acc = acc.add(ch[key].value_counts(), fill_value=0)
    return acc.astype("int64")

subtables = ["bureau.csv", "previous_application.csv", "POS_CASH_balance.csv",
             "credit_card_balance.csv", "installments_payments.csv"]

rows = []
for f in subtables:
    c = scan_counts(f)
    rows.append({
        "sub_table": f.replace(".csv", ""),
        "applicants_covered": c.size,
        "coverage_%": round(c.size / n_msme * 100, 1),
        "total_rows": int(c.sum()),
        "median_rows_per_applicant": int(c.median()) if c.size else 0,
    })

# bureau_balance only links through bureau, so build an SK_ID_BUREAU -> SK_ID_CURR
# lookup first, restricted to MSME applicants
bmap = {}
for ch in pd.read_csv(os.path.join(DATA_DIR, "bureau.csv"),
                      usecols=["SK_ID_CURR", "SK_ID_BUREAU"], chunksize=CHUNK):
    ch = ch[ch["SK_ID_CURR"].isin(msme_ids)]
    bmap.update(dict(zip(ch["SK_ID_BUREAU"], ch["SK_ID_CURR"])))

bb = pd.Series(dtype="int64")
for ch in pd.read_csv(os.path.join(DATA_DIR, "bureau_balance.csv"),
                      usecols=["SK_ID_BUREAU"], chunksize=CHUNK):
    mapped = ch["SK_ID_BUREAU"].map(bmap).dropna()
    if len(mapped):
        bb = bb.add(mapped.value_counts(), fill_value=0)

rows.append({
    "sub_table": "bureau_balance",
    "applicants_covered": bb.size,
    "coverage_%": round(bb.size / n_msme * 100, 1),
    "total_rows": int(bb.sum()),
    "median_rows_per_applicant": int(bb.median()) if bb.size else 0,
})

coverage = pd.DataFrame(rows).set_index("sub_table")
coverage.to_csv(os.path.join(OUT_DIR, "subtable_coverage.csv"))
print(f"MSME proxy applicants: {n_msme:,}\n")
print(coverage.to_string())


MSME proxy applicants: 38,412

                       applicants_covered  coverage_%  total_rows  median_rows_per_applicant
sub_table                                                                                   
bureau                              31584        82.2      155675                          4
previous_application                37083        96.5      183106                          4
POS_CASH_balance                    36827        95.9     1057554                         22
credit_card_balance                 12095        31.5      394189                         18
installments_payments               37130        96.7     1442792                         25
bureau_balance                      11228        29.2     1547388                         92


In [12]:
fig, ax = plt.subplots(figsize=(7, 4.2))
cov = coverage.sort_values("coverage_%")
ax.barh(cov.index, cov["coverage_%"], color="#5B8C7B", edgecolor="black", linewidth=0.4)
for y, v in enumerate(cov["coverage_%"]):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Share of MSME-proxy applicants with ≥ 1 record (%)")
ax.set_title("Relational sub-table coverage of the MSME proxy")
ax.set_xlim(0, 105)
sns.despine(ax=ax)
savefig(fig, "eda_07_subtable_coverage.png",
        "Proportion of the 38,412 MSME-proxy applicants that appear in each relational "
        "sub-table. Coverage determines which behavioural feature families can be built "
        "in Stage 2.")


saved -> outputs\eda_07_subtable_coverage.png   |  Proportion of the 38,412 MSME-proxy applicants that appear in each relational sub-table. Coverage determines which behavioural feature families can be built in Stage 2.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\eda_07_subtable_coverage.png'

## 9. Stage 1 summary

What I'm carrying forward into Stage 2:

- **Severe class imbalance** - 8.1% default in the full population, 10.2% in the
  MSME proxy (≈ 8.8 : 1). This is why I'll need SMOTE on the training partition
  and why default-class recall, not accuracy, is the metric that matters.
- **MSME proxy** - `ORGANIZATION_TYPE == "Self-employed"`, 38,412 applicants,
  saved to `outputs/msme_proxy.csv`.
- **Missingness** is mostly in building-description fields and `EXT_SOURCE_1`;
  most of the fields I'll actually model with are fairly complete.
- **`EXT_SOURCE_2/3`** are the strongest single raw predictors - defaulters score
  lower on all three external scores.
- **Sub-table coverage** tells me which behavioural feature families are actually
  viable for the MSME proxy (full numbers in `outputs/subtable_coverage.csv`).


In [13]:
# dump a quick index of every figure this notebook produced, for the List of Figures
fig_index = pd.Series(FIG_INDEX, name="caption").rename_axis("file").reset_index()
fig_index.to_csv(os.path.join(OUT_DIR, "eda_figure_index.csv"), index=False)
print("Figures written to outputs/ this run:\n")
for f, cap in FIG_INDEX.items():
    print(f"  {f}\n      {cap}\n")


Figures written to outputs/ this run:

  eda_01_class_imbalance_full.png
      Class distribution of the target variable in the full application population (n = 307,511); defaults represent 8.1% of cases.

  eda_02_default_rate_by_organisation.png
      Default rate by organisation type for categories with at least 2,000 applicants. The Self-employed group (highlighted) is the MSME proxy adopted for this study.

  eda_03_missingness_top25.png
      The 25 features with the highest proportion of missing values in the MSME proxy sub-population. Building-information and external-score fields dominate.

  eda_04_key_distributions.png
      Kernel-density comparison of applicant age, employment length, annual income (log10) and requested credit amount (log10) between repaid and defaulted loans in the MSME proxy.

  eda_05_ext_source_distributions.png
      Distribution of the three external credit scores by default status in the MSME proxy. Defaulted applicants show consistently lower score